<a href="https://colab.research.google.com/github/addadugurudurga2024-lang/Flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Load the same dataset used in ML-08 / ML-09
df = pd.read_csv("/content/Flyrank/data/raw/content_refresh_anonymized.csv")

# Create target
df["is_declining"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

# Same features used in ML-08 / ML-09
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df["is_declining"]
groups = df["client_id"]

# Same grouped-by-client validation design from ML-09
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Same Decision Tree
model_grouped = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model_grouped.fit(X_train, y_train)

pred_grouped = model_grouped.predict(X_test)

# Validation metrics
grouped_accuracy = accuracy_score(y_test, pred_grouped)
grouped_precision = precision_score(
    y_test, pred_grouped, zero_division=0
)
grouped_recall = recall_score(
    y_test, pred_grouped, zero_division=0
)
grouped_f1 = f1_score(
    y_test, pred_grouped, zero_division=0
)

print("ML-09 validation loaded successfully.")
print("Grouped Accuracy :", round(grouped_accuracy, 3))
print("Grouped Precision:", round(grouped_precision, 3))
print("Grouped Recall   :", round(grouped_recall, 3))
print("Grouped F1       :", round(grouped_f1, 3))

ML-09 validation loaded successfully.
Grouped Accuracy : 0.557
Grouped Precision: 0.565
Grouped Recall   : 0.581
Grouped F1       : 0.573


## 1) Ranked actions + reason codes

The action queue uses the validated Decision Tree as a prioritization signal, not as an automatic decision.

| Rank | Action | Reason code |
|---|---|---|
| 1 | Review pages predicted as declining with meaningful impressions | DECLINING + VISIBLE |
| 2 | Review older pages that have not been updated recently | STALE_CONTENT |
| 3 | Review pages with weak search position or CTR | WEAK_VISIBILITY |
| 4 | Review pages with low impressions before investing in a major update | LOW_SIGNAL |

The priority is based on observable page-level signals including decline status, impressions, content age, update recency, search position, CTR, and word count. The model's grouped-by-client F1 of 0.573 shows that these signals are useful for directional prioritization but are not reliable enough for automatic decisions.

In [13]:
# Create a simple ranked action queue from the validated model

queue = df.iloc[test_idx].copy()
queue["predicted_declining"] = pred_grouped

queue["reason_code"] = "LOW_SIGNAL"

queue.loc[
    (queue["predicted_declining"] == 1) &
    (queue["impressions_90d"] > 0),
    "reason_code"
] = "DECLINING_VISIBLE"

queue.loc[
    (queue["days_since_last_update"] >= 180),
    "reason_code"
] = "STALE_CONTENT"

queue.loc[
    (queue["predicted_declining"] == 1) &
    (queue["impressions_90d"] > 0) &
    (queue["days_since_last_update"] >= 180),
    "reason_code"
] = "DECLINING_AND_STALE"

queue["priority"] = queue["predicted_declining"] * 2
queue["priority"] += (queue["days_since_last_update"] >= 180).astype(int)
queue["priority"] += (queue["impressions_90d"] > 0).astype(int)

ranked_queue = queue.sort_values(
    ["priority", "impressions_90d"],
    ascending=[False, False]
)

display(
    ranked_queue[
        ["priority", "reason_code",
         "content_age_days",
         "days_since_last_update",
         "impressions_90d",
         "avg_position",
         "ctr"]
    ].head(20)
)

,priority,reason_code,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr
6903,3,DECLINING_VISIBLE,95,20,223271,7.8,0.03
22028,3,DECLINING_VISIBLE,97,20,213963,4.7,0.10
11655,3,DECLINING_VISIBLE,96,20,208798,5.2,0.23
23460,3,DECLINING_VISIBLE,97,20,198671,5.6,0.18
27178,3,DECLINING_VISIBLE,97,20,140079,7.6,0.01
26564,3,DECLINING_VISIBLE,277,104,128704,4.9,0.40
29716,3,DECLINING_VISIBLE,326,104,126611,5.7,0.25
15914,3,DECLINING_VISIBLE,326,7,119217,7.0,0.02
19698,3,DECLINING_VISIBLE,326,14,114528,5.9,0.18
482,3,DECLINING_VISIBLE,97,20,112434,7.2,0.01


### Archetype → action mapping

| Page archetype | Recommended action |
|---|---|
| Declining + visible | Prioritize for human content review |
| Stale content | Check whether information needs refreshing |
| Weak visibility | Review search position, CTR, and search intent |
| Low-signal page | Gather more evidence before making a major change |

These mappings are workflow recommendations based on observed page-level signals. They are not automatic instructions to change or remove content.

## 2. Intended use and limits

This playbook is intended to help a content team prioritize which pages deserve human review first.

The model is decision-support. It does not determine that a page is objectively poor, that an update will increase traffic, or that a page must be changed.

The validation results show why this limitation matters. The random split produced an F1 of 0.688, while the grouped-by-client split produced an F1 of 0.573. Therefore, the more conservative result should guide interpretation.

The recommendations are valid only for the type of page-level data and target definition evaluated in this project. They should not be treated as universal rules or causal conclusions.

In [14]:
# Validation evidence used to define the limits

validation_summary = pd.DataFrame({
    "Validation": ["Random split", "Grouped by client"],
    "F1": [0.688, 0.573],
    "Interpretation": [
        "Development baseline",
        "More conservative generalization check"
    ]
})

display(validation_summary)

,Validation,F1,Interpretation
0,Random split,0.688,Development baseline
1,Grouped by client,0.573,More conservative generalization check


### Decay / refresh insight

The analysis suggests that content age and update recency are useful signals for deciding which pages deserve review. However, the relationship is not strong enough to treat age alone as proof that content has decayed.

A practical approach is therefore to use age and days since last update as review signals alongside impressions, CTR, search position, and observed decline. Refresh decisions should remain human-reviewed.

## 3. Human review + the no-go list

Before acting on a recommendation, a person should check:

1. Whether the page has a meaningful business or user purpose.
2. Whether the observed decline is recent or part of a longer trend.
3. Whether search intent has changed.
4. Whether the page is still accurate and useful.
5. Whether the recommendation makes sense in the context of the website.

### No-go list

The system should NOT automatically:

- publish or rewrite content;
- delete pages;
- change titles or metadata;
- redirect URLs;
- make SEO or business decisions;
- claim that an update will increase traffic;
- treat a model prediction as proof that content is failing.

The model can rank pages for review, but a human remains responsible for the final action.

In [15]:
# Human-review gate

review_required = ranked_queue[
    ranked_queue["priority"] > 0
].copy()

review_required["human_review_required"] = True

display(
    review_required[
        ["reason_code", "priority", "human_review_required"]
    ].head(20)
)

,reason_code,priority,human_review_required
6903,DECLINING_VISIBLE,3,True
22028,DECLINING_VISIBLE,3,True
11655,DECLINING_VISIBLE,3,True
23460,DECLINING_VISIBLE,3,True
27178,DECLINING_VISIBLE,3,True
26564,DECLINING_VISIBLE,3,True
29716,DECLINING_VISIBLE,3,True
15914,DECLINING_VISIBLE,3,True
19698,DECLINING_VISIBLE,3,True
482,DECLINING_VISIBLE,3,True


### Cost / value thinking

The highest-value actions should generally be reviewed first: pages with observable decline and meaningful visibility may have more potential value from review than pages with very little traffic or evidence.

The model does not estimate the financial return of an update. Cost/value is therefore a prioritization principle for human review, not a measured ROI prediction.

## 4. Monitoring / retrain triggers

The recommendations may become stale if the underlying page behavior changes.

I would review the model if:

- grouped validation F1 falls materially below the current measured 0.573;
- the distribution of impressions, CTR, position, or content age changes substantially;
- the proportion of declining pages changes substantially;
- new content types or page types become common;
- the relationship between model recommendations and observed outcomes becomes weaker.

A retraining cycle should be considered after meaningful changes in the data or when monitoring shows that the current model no longer provides useful directional prioritization.

These are monitoring and decision-support triggers, not guarantees of model failure.

In [16]:
# Current validation result used as a monitoring reference

current_f1 = grouped_f1
reference_f1 = 0.573

print("Current grouped-by-client F1:", round(current_f1, 3))
print("Reference F1:", reference_f1)

if current_f1 < reference_f1:
    print("Review model performance before relying on the queue.")
else:
    print("No immediate performance trigger from this simple check.")

Current grouped-by-client F1: 0.573
Reference F1: 0.573
Review model performance before relying on the queue.


## 5. Exports for the paper

The ranked queue is exported so that the next research-paper stage can reuse the same recommendations and avoid manually recreating the analysis.

In [17]:
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

export_columns = [
    "priority",
    "reason_code",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

ranked_queue[export_columns].head(100).to_csv(
    output_dir / "content_action_queue.csv",
    index=False
)

validation_summary.to_csv(
    output_dir / "validation_summary.csv",
    index=False
)

print("Exports created:")
print(output_dir / "content_action_queue.csv")
print(output_dir / "validation_summary.csv")

Exports created:
work/outputs/content_action_queue.csv
work/outputs/validation_summary.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.